In [5]:
!pip install ijson==3.2.1

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import ijson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from glob import glob

# loading in raw xsens data

In [8]:
def load_files_to_dfs(folder_path):
    # List to store the dataframes
    honda_dfs = []

    # Get all folders (assuming they are numbered and in the folder_path)
    for folder in os.listdir(folder_path):
        folder_num = folder  # Folder name (assuming it is the number)
        folder_full_path = os.path.join(folder_path, folder)

        if os.path.isdir(folder_full_path):
            # Look specifically for the 'xsens.csv' and 'labels.csv' file in each folder
            file_path_xsens = os.path.join(folder_full_path, 'xsens.csv')
            file_path_labels = os.path.join(folder_full_path, 'labels.csv')

            # making sure both paths exist prior to uploading df
            if os.path.exists(file_path_xsens):
                if os.path.exists(file_path_labels):
                    # Load the files into a DataFrames (assuming csv format)
                    df_xsens = pd.read_csv(file_path_xsens)
                    df_labels = pd.read_csv(file_path_labels)

                    # Merge the two DataFrames on the 'time', 'participant_id', 'task' column
                    df = pd.merge(df_xsens, df_labels, on=['time', 'participant_id', 'task'])

                    # Add participant_num column
                    df['participant_num'] = folder_num
                    print(folder_num)

                    # Add the DataFrame to the list
                    honda_dfs.append(df)

    return honda_dfs

# Usage example
folder_path = '/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB'  # Replace with the path to the main directory containing folders
dataframes = load_files_to_dfs(folder_path)

# # Print the dataframes list
# for idx, df in enumerate(dataframes):
#     print(f"DataFrame from folder {idx + 1}:\n", df.head())


id15
id01
id12
id14
id24
id23
id08
id07
id25
id22
id19
id02
id04
id10
id16
id18
id17
id05
id13
id03


In [9]:
len(dataframes)

20

In [10]:
dataframes_copy = dataframes

In [11]:
# list(dataframes_copy[0].columns)

In [12]:
dataframes[0]['walk_mode'].unique()

array(['walk', 'slope_down', 'slope_up', 'stairs_up'], dtype=object)

In [13]:
sensor_types = ['orientation','position','velocity','acceleration','angularVelocity','angularAcceleration','sensorFreeAcceleration','sensorMagneticField','sensorOrientation','jointAngle','jointAngleXZY']

sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']

axes = ['x', 'y', 'z', 'ql', 'qi', 'qj', 'qk']

columns = ['walk_mode', 'time', 'participant_id', 'task', 'sensor_location', 'orientation_q1', 'orientation_qi', 'orientation_qj', 'orientation_qk',
           'position_x', 'position_y', 'position_z', 'velocity_x', 'velocity_y', 'velocity_z', 'acceleration_x', 'acceleration_y', 'acceleration_z',
           'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z', 'angularAcceleration_x', 'angularAcceleration_y', 'angularAcceleration_z',
           'sensorFreeAcceleration_x', 'sensorFreeAcceleration_y', 'sensorFreeAcceleration_z', 'sensorMagneticField_x', 'sensorMagneticField_y',
           'sensorMagneticField_z', 'sensorOrientation_q1', 'sensorOrientation_qi', 'sensorOrientation_qj', 'sensorOrientation_qk', 'jointAngle_x',
           'jointAngle_y', 'jointAngle_z', 'jointAngleXZY_x', 'jointAngleXZY_y', 'jointAngleXZY_z']


In [14]:
new_column_names = {
    'index': 'index',
    'orientation_q1': 'OriInc_q0',                      # omit in future
    'orientation_qi': 'OriInc_q1',                      # omit in future
    'orientation_qj': 'OriInc_q2',                      # omit in future
    'orientation_qk': 'OriInc_q3',                      # omit in future
    'sensor_location': 'sensor_location',
    'position_x': 'Position_X',                         # omit in future
    'position_y': 'Position_Y',                         # omit in future
    'position_z': 'Position_Z',                         # omit in future
    'velocity_x': 'VelInc_X',                           # omit in future
    'velocity_y': 'VelInc_Y',                           # omit in future
    'velocity_z': 'VelInc_Z',                           # omit in future
    'acceleration_x': 'Acc_X',
    'acceleration_y': 'Acc_Y',
    'acceleration_z': 'Acc_Z',
    'angularVelocity_x': 'Gyr_X',
    'angularVelocity_y': 'Gyr_Y',
    'angularVelocity_z': 'Gyr_Z',
    'angularAcceleration_x': 'AngularAcceleration_X',   # omit in future
    'angularAcceleration_y': 'AngularAcceleration_Y',   # omit in future
    'angularAcceleration_z': 'AngularAcceleration_Z',   # omit in future
    'sensorFreeAcceleration_x': 'FreeAcc_X',
    'sensorFreeAcceleration_y': 'FreeAcc_Y',
    'sensorFreeAcceleration_z': 'FreeAcc_Z',
    'sensorMagneticField_x': 'Mag_X',
    'sensorMagneticField_y': 'Mag_Y',
    'sensorMagneticField_z': 'Mag_Z',
    'sensorOrientation_q1': 'sensorOrientation_q1',     # omit in future
    'sensorOrientation_qi': 'sensorOrientation_qi',     # omit in future
    'sensorOrientation_qj': 'sensorOrientation_qj',     # omit in future
    'sensorOrientation_qk': 'sensorOrientation_qk',     # omit in future
    'jointAngle_x': 'JointAngle_X',                     # omit in future
    'jointAngle_y': 'JointAngle_Y',                     # omit in future
    'jointAngle_z': 'JointAngle_Z',                     # omit in future
    'jointAngleXZY_x': 'Roll',                          # omit in future
    'jointAngleXZY_y': 'Pitch',                         # omit in future
    'jointAngleXZY_z': 'Yaw'                            # omit in future
}

In [15]:
# get the 5 types of surfaces/actions done in this dataset
walk_modes = dataframes_copy[0]['walk_mode'].unique()

# create a subset dataframe for each of the types
subset_dfs = {}
for walk_mode in walk_modes:
  for df in dataframes_copy:
      identifier = walk_mode + '_' + df['participant_num'].iloc[0]
      if identifier not in subset_dfs:
          subset_dfs[identifier] = df[df['walk_mode'] == walk_mode]


In [16]:
# subset_dfs.keys()

In [17]:
subset_dfs['slope_up_id12']

,time,participant_id,task,orientation_Pelvis_q1,orientation_Pelvis_qi,orientation_Pelvis_qj,orientation_Pelvis_qk,orientation_L5_q1,orientation_L5_qi,orientation_L5_qj,...,insoles_RightFoot_time_to_lift,xsens_footContacts_LeftFoot_Heel,xsens_footContacts_LeftFoot_Toe,xsens_footContacts_RightFoot_Heel,xsens_footContacts_RightFoot_Toe,walk_orientation,walk_interaction,walk_mode_id,walk_orientation_id,participant_num
3369,56150,12,B,-0.997874,-0.015332,-0.004589,0.063177,-0.994699,-0.012586,-0.026812,...,633.0,False,False,True,False,curve_left,no,3,6,id12
3370,56167,12,B,-0.997711,-0.003879,-0.003003,0.067449,-0.994729,-0.006569,-0.024187,...,616.0,False,False,True,False,curve_left,no,3,6,id12
3371,56183,12,B,-0.997551,0.007205,-0.001577,0.069558,-0.994843,-0.000658,-0.022280,...,600.0,False,False,True,False,curve_left,no,3,6,id12
3372,56200,12,B,-0.997422,0.015958,-0.000924,0.069950,-0.995018,0.004017,-0.021477,...,583.0,False,False,True,False,curve_left,no,3,6,id12
3373,56217,12,B,-0.997367,0.021087,0.000059,0.069392,-0.995250,0.006587,-0.020977,...,566.0,False,False,True,False,curve_left,no,3,6,id12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30948,515800,12,B,-0.997126,-0.011914,-0.000285,-0.074821,-0.993817,-0.009327,-0.035550,...,217.0,False,False,False,False,turn_around_clockwise,no,3,8,id12
30949,515817,12,B,-0.997331,-0.014365,-0.000540,-0.071590,-0.994009,-0.011500,-0.035017,...,200.0,False,False,False,False,turn_around_clockwise,no,3,8,id12
30950,515833,12,B,-0.997635,-0.016925,-0.003531,-0.066519,-0.994261,-0.013871,-0.035524,...,184.0,True,False,False,False,turn_around_clockwise,no,3,8,id12
30951,515850,12,B,-0.998140,-0.019433,-0.004947,-0.057572,-0.994757,-0.016214,-0.034882,...,167.0,True,False,False,False,turn_around_clockwise,no,3,8,id12


In [18]:
import pandas as pd

# Define your list of sensor locations
sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand',
                    'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg', 'RightFoot',
                    'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']

# Function to extract the sensor location from a column name based on the sensor_locations list
def extract_sensor_location(column_name):
    for location in sensor_locations:
        if location in column_name:
            return location
    return None

# Function to remove the sensor location from a column name
def remove_sensor_location(column_name):
    for location in sensor_locations:
        if location in column_name:
            # Remove the sensor location from the column name
            return column_name.replace(location, '').strip('_')
    return column_name

# Loop over each DataFrame in subset_dfs (assuming it's a dictionary of trial DataFrames)
for key, trial_df in subset_dfs.items():
    df_list = []
    surface_type = key.split('_')
    if len(surface_type) > 2:
        surface_type = ' '.join(surface_type[:2])
    else:
        surface_type = surface_type[0]

    # For each sensor, collect relevant columns and create a new DataFrame
    for sensor_location in sensor_locations:
        # Find all columns related to this sensor location
        sensor_columns = [col for col in trial_df.columns if sensor_location in col]

        if sensor_columns:
            # Create a new DataFrame for this sensor
            df_by_sensor = pd.DataFrame({
                "time": trial_df["time"],
                "participant_id": trial_df["participant_id"],
                "task": trial_df["task"],
                "surface_type": surface_type,  # Use the derived surface type
                "sensor_location": sensor_location
            })

            # For each sensor-related column, add it to the new DataFrame
            for col in sensor_columns:
                # Remove sensor location from the column name for readability
                new_column_name = remove_sensor_location(col)
                df_by_sensor[new_column_name] = trial_df[col]

            # Append the sensor DataFrame to the list
            df_list.append(df_by_sensor)

    # Concatenate all DataFrames for this trial, if any were created
    if df_list:
        df = pd.concat(df_list)

        # Export df to CSV
        df.to_csv(f'/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/{key}.csv', index=False)

        # Clear the local memory
        # del df_list
        # del df

        print(f"Processed {key}")
    else:
        print(f"No valid sensor columns to process for {key}")


Processed walk_id15
Processed walk_id01
Processed walk_id12
Processed walk_id14
Processed walk_id24
Processed walk_id23
Processed walk_id08
Processed walk_id07
Processed walk_id25
Processed walk_id22
Processed walk_id19
Processed walk_id02
Processed walk_id04
Processed walk_id10
Processed walk_id16
Processed walk_id18
Processed walk_id17
Processed walk_id05
Processed walk_id13
Processed walk_id03
Processed slope_down_id15
Processed slope_down_id01
Processed slope_down_id12
Processed slope_down_id14
Processed slope_down_id24
Processed slope_down_id23
Processed slope_down_id08
Processed slope_down_id07
Processed slope_down_id25
Processed slope_down_id22
Processed slope_down_id19
Processed slope_down_id02
Processed slope_down_id04
Processed slope_down_id10
Processed slope_down_id16
Processed slope_down_id18
Processed slope_down_id17
Processed slope_down_id05
Processed slope_down_id13
Processed slope_down_id03
Processed slope_up_id15
Processed slope_up_id01
Processed slope_up_id12
Processe

In [19]:
# # this is all the data for participant 1 from course A
# ['slope_down_id01']
# ['walk_id01']
# ['slope_up_id01']
# ['stairs_down_id01']
# ['stairs_up_id01']

In [20]:
subset_dfs.keys()

dict_keys(['walk_id15', 'walk_id01', 'walk_id12', 'walk_id14', 'walk_id24', 'walk_id23', 'walk_id08', 'walk_id07', 'walk_id25', 'walk_id22', 'walk_id19', 'walk_id02', 'walk_id04', 'walk_id10', 'walk_id16', 'walk_id18', 'walk_id17', 'walk_id05', 'walk_id13', 'walk_id03', 'slope_down_id15', 'slope_down_id01', 'slope_down_id12', 'slope_down_id14', 'slope_down_id24', 'slope_down_id23', 'slope_down_id08', 'slope_down_id07', 'slope_down_id25', 'slope_down_id22', 'slope_down_id19', 'slope_down_id02', 'slope_down_id04', 'slope_down_id10', 'slope_down_id16', 'slope_down_id18', 'slope_down_id17', 'slope_down_id05', 'slope_down_id13', 'slope_down_id03', 'slope_up_id15', 'slope_up_id01', 'slope_up_id12', 'slope_up_id14', 'slope_up_id24', 'slope_up_id23', 'slope_up_id08', 'slope_up_id07', 'slope_up_id25', 'slope_up_id22', 'slope_up_id19', 'slope_up_id02', 'slope_up_id04', 'slope_up_id10', 'slope_up_id16', 'slope_up_id18', 'slope_up_id17', 'slope_up_id05', 'slope_up_id13', 'slope_up_id03', 'stairs_u

In [21]:
import os
import shutil

# Iterate over subset_dfs items
for key, value in subset_dfs.items():
    # Extract participant ID
    participant = key.split('_')[-1]
    participant_number = participant[-2:]

    # Define the directory for this participant
    participant_dir = f'/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/Participant_{participant_number}'

    # Create the participant folder if it doesn't exist
    os.makedirs(participant_dir, exist_ok=True)

    # Define the source CSV file path
    source_file = f'/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/{key}.csv'

    # Define the destination file path in the participant's folder
    destination_file = os.path.join(participant_dir, f'{key}.csv')

    # Move the file if it exists
    if os.path.exists(source_file):
        shutil.move(source_file, destination_file)
        print(f"Moved {source_file} to {destination_file}")
    else:
        print(f"{source_file} does not exist")


Moved /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/walk_id15.csv to /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/Participant_15/walk_id15.csv
Moved /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/walk_id01.csv to /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/Participant_01/walk_id01.csv
Moved /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/walk_id12.csv to /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/Participant_12/walk_id12.csv
Moved /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/walk_id14.csv to /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/Participant_14/walk_id14.csv
Moved /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/walk_id24.csv to /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/Participant_24/walk_id24.csv
Moved /content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/walk_id23.csv to /content/drive/MyDrive/Next_Step/Honda_Participants/Cours

# unless pre preprocessing changes, no need to run above

In [22]:

# Define the folder path containing your subfolders with CSV files
folder_path = '/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB'  # Replace with your actual folder path

# Create an empty dictionary to store the merged DataFrames
merged_df_dict = {}

# Get a list of all subdirectories within the main folder
subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()]

# Function to merge CSV files in a single subfolder
def merge_csv_files_in_folder(subfolder_path):
    # Get all CSV files in the subfolder
    csv_files = glob(os.path.join(subfolder_path, '*.csv'))

    # If there are CSV files in the folder, read and merge them
    if csv_files:
        # Read each CSV file and store them in a list
        df_list = [pd.read_csv(file, low_memory=False) for file in csv_files]

        # Merge the DataFrames
        merged_df = pd.concat(df_list, ignore_index=True)

        # Extract folder name to create a key for the merged DataFrame
        folder_name = os.path.basename(subfolder_path)

        # Return the merged DataFrame with folder name as the key
        print(f"Merged {len(csv_files)} CSV files in folder: {folder_name}")
        return folder_name, merged_df
    else:
        print(f"No CSV files found in {subfolder_path}")
        return None

for subfolder in subfolders:
    result = merge_csv_files_in_folder(subfolder)
    if result:
        key, merged_df = result
        merged_df_dict[key] = merged_df

for key, df in merged_df_dict.items():
    df.to_csv(f'/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB/{key}_merged.csv', index=False)

print(f"Processed {len(merged_df_dict)} folders and merged their CSV files.")

Merged 4 CSV files in folder: Participant_15
Merged 4 CSV files in folder: Participant_01
Merged 4 CSV files in folder: Participant_12
Merged 4 CSV files in folder: Participant_14
Merged 4 CSV files in folder: Participant_24
Merged 4 CSV files in folder: Participant_23
Merged 4 CSV files in folder: Participant_08
Merged 4 CSV files in folder: Participant_07
Merged 4 CSV files in folder: Participant_25
Merged 4 CSV files in folder: Participant_22
Merged 4 CSV files in folder: Participant_19
Merged 4 CSV files in folder: Participant_02
Merged 4 CSV files in folder: Participant_04
Merged 4 CSV files in folder: Participant_10
Merged 4 CSV files in folder: Participant_16
Merged 4 CSV files in folder: Participant_18
Merged 4 CSV files in folder: Participant_17
Merged 4 CSV files in folder: Participant_05
Merged 4 CSV files in folder: Participant_13
Merged 4 CSV files in folder: Participant_03
Processed 20 folders and merged their CSV files.


In [23]:
def load_files_to_dfs(folder_path):
    # List to store the dataframes
    honda_dfs = []

    # Get all folders (assuming they are numbered and in the folder_path)
    for folder in os.listdir(folder_path):
        folder_num = folder  # Folder name (assuming it is the number)
        folder_full_path = os.path.join(folder_path, folder)

        if os.path.isdir(folder_full_path):
            # Look specifically for the 'insoles.csv' and 'labels.csv' file in each folder
            file_path_insoles = os.path.join(folder_full_path, 'insoles.csv')
            file_path_labels = os.path.join(folder_full_path, 'labels.csv')

            # making sure both paths exist prior to uploading df
            if os.path.exists(file_path_insoles):
                if os.path.exists(file_path_labels):
                    # Load the files into a DataFrames (assuming csv format)
                    df_insoles = pd.read_csv(file_path_insoles)
                    df_labels = pd.read_csv(file_path_labels)

                    # Merge the two DataFrames on the 'time', 'participant_id', 'task' column
                    df = pd.merge(df_insoles, df_labels, on=['time', 'participant_id', 'task'])

                    # Add participant_num column
                    df['participant_num'] = folder_num
                    print(folder_num)

                    # Add the DataFrame to the list
                    honda_dfs.append(df)

    return honda_dfs

# Usage example
folder_path = '/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set/courseB'  # Replace with the path to the main directory containing folders
dataframes = load_files_to_dfs(folder_path)

# # Print the dataframes list
# for idx, df in enumerate(dataframes):
#     print(f"DataFrame from folder {idx + 1}:\n", df.head())


id15
id01
id12
id14
id24
id23
id08
id07
id25
id22
id19
id02
id04
id10
id16
id18
id17
id05
id13
id03


In [24]:
len(dataframes)

20

In [25]:
dataframes_copy = dataframes

In [26]:
#list all the columns
# list(dataframes[0].columns)

In [27]:
dataframes[0]['walk_mode'].unique()

array(['walk', 'slope_down', 'slope_up', 'stairs_up'], dtype=object)

In [28]:
# dataframes[0]

In [29]:
sensor_types = ['Acc','Mag','Gyr']

leg = ['Left', 'Right']

sensor_locations = ['Hallux', 'Toes', 'Met1', 'Met3', 'Met5', 'Arch', 'Heel_R', 'Heel_L']

axes = ['x', 'y', 'z', 'raw', 'norm']

columns = ['Unnamed: 0', 'time', 'participant_id', 'task', 'Left_Hallux', 'Left_Toes', 'Left_Met1',
           'Left_Met3', 'Left_Met5', 'Left_Arch', 'Left_Heel_R', 'Left_Heel_L', 'Left_Acc_x', 'Left_Acc_y', 'Left_Acc_z',
           'Left_Gyr_x', 'Left_Gyr_y', 'Left_Gyr_z', 'Left_Mag_x', 'Left_Mag_y', 'Left_Mag_z', 'Left_Temp', 'Right_Hallux',
           'Right_Toes', 'Right_Met1', 'Right_Met3', 'Right_Met5', 'Right_Arch', 'Right_Heel_L', 'Right_Heel_R', 'Right_Acc_x',
           'Right_Acc_y', 'Right_Acc_z', 'Right_Gyr_x', 'Right_Gyr_y', 'Right_Gyr_z', 'Right_Mag_x', 'Right_Mag_y', 'Right_Mag_z',
           'Right_Temp', 'Right_Toes_raw', 'Right_Hallux_raw', 'Right_Met5_raw', 'Right_Met3_raw', 'Right_Met1_raw', 'Right_Arch_raw', 'Right_Heel_R_raw',
           'Right_Heel_L_raw', 'Left_Toes_raw', 'Left_Hallux_raw', 'Left_Met5_raw', 'Left_Met3_raw', 'Left_Met1_raw', 'Left_Arch_raw', 'Left_Heel_R_raw',
           'Left_Heel_L_raw', 'Right_Acc_x_raw', 'Right_Acc_y_raw', 'Right_Acc_z_raw', 'Right_Mag_x_raw', 'Right_Mag_y_raw', 'Right_Mag_z_raw', 'Right_Gyr_x_raw',
           'Right_Gyr_y_raw', 'Right_Gyr_z_raw', 'Left_Acc_x_raw', 'Left_Acc_y_raw', 'Left_Acc_z_raw', 'Left_Mag_x_raw', 'Left_Mag_z_raw', 'Left_Gyr_x_raw',
           'Left_Gyr_y_raw', 'Left_Gyr_z_raw', 'Left_Hallux_norm', 'Left_Toes_norm', 'Left_Met1_norm', 'Left_Met3_norm', 'Left_Met5_norm', 'Left_Arch_norm',
           'Left_Heel_R_norm', 'Left_Heel_L_norm', 'Right_Hallux_norm', 'Right_Toes_norm', 'Right_Met1_norm', 'Right_Met3_norm', 'Right_Met5_norm', 'Right_Arch_norm',
           'Right_Heel_L_norm', 'Right_Heel_R_norm', 'Left_Max_Pressure_norm', 'Right_Max_Pressure_norm', 'walk_mode', 'repetition', 'insoles_RightFoot_is_step',
           'insoles_LeftFoot_is_step', 'insoles_RightFoot_is_lifted', 'insoles_LeftFoot_is_lifted', 'insoles_LeftFoot_on_ground',
           'insoles_RightFoot_on_ground', 'insoles_LeftFoot_time_to_step', 'insoles_LeftFoot_time_to_lift', 'insoles_RightFoot_time_to_step',
           'insoles_RightFoot_time_to_lift', 'xsens_footContacts_LeftFoot_Heel', 'xsens_footContacts_LeftFoot_Toe', 'xsens_footContacts_RightFoot_Heel',
           'xsens_footContacts_RightFoot_Toe', 'walk_orientation', 'walk_interaction', 'walk_mode_id', 'walk_orientation_id', 'participant_num']


In [30]:
# get the 5 types of surfaces/actions done in this dataset
walk_modes = dataframes_copy[0]['walk_mode'].unique()

# create a subset dataframe for each of the types
subset_dfs = {}
for walk_mode in walk_modes:
  for df in dataframes_copy:
      identifier = walk_mode + '_' + df['participant_num'].iloc[0]
      if identifier not in subset_dfs:
          subset_dfs[identifier] = df[df['walk_mode'] == walk_mode]


In [31]:
# change the column names wiht no suffix to have

In [32]:
# subset_dfs['slope_up_id12']

In [33]:
extras_df = []

for key, df in subset_dfs.items():
    df.columns = columns
    # keep only listed columns
    df = df[['Unnamed: 0', 'time', 'participant_id', 'task','Left_Max_Pressure_norm', 'Right_Max_Pressure_norm', 'walk_mode', 'repetition', 'insoles_RightFoot_is_step',
           'insoles_LeftFoot_is_step', 'insoles_RightFoot_is_lifted', 'insoles_LeftFoot_is_lifted', 'insoles_LeftFoot_on_ground',
           'insoles_RightFoot_on_ground', 'insoles_LeftFoot_time_to_step', 'insoles_LeftFoot_time_to_lift', 'insoles_RightFoot_time_to_step',
           'insoles_RightFoot_time_to_lift', 'xsens_footContacts_LeftFoot_Heel', 'xsens_footContacts_LeftFoot_Toe', 'xsens_footContacts_RightFoot_Heel',
           'xsens_footContacts_RightFoot_Toe', 'walk_orientation', 'walk_interaction']]

    extras_df.append(df)

# Concatenate all trials into a single DataFrame
all_foot_sensor_extras_df = pd.concat(extras_df, ignore_index=True)

In [34]:
# all_foot_sensor_extras_df

In [35]:
import pandas as pd

# Define your list of sensor locations
sensor_locations = ['Hallux', 'Toes', 'Met1', 'Met3', 'Met5', 'Arch', 'Heel_R', 'Heel_L']

# Function to extract the sensor location from a column name based on the sensor_locations list
def extract_sensor_location(column_name):
    for location in sensor_locations:
        if location in column_name:
            return location
    return None

# Function to remove the sensor location from a column name
def remove_sensor_location(column_name):
    for location in sensor_locations:
        if location in column_name:
            return column_name.replace(location, '').strip('_')
    return column_name

# Initialize a list to hold DataFrames for all trials
all_trials_df_list = []

# Loop over each DataFrame in subset_dfs (assuming it's a dictionary of trial DataFrames)
for key, trial_df in subset_dfs.items():
    df_list = []
    surface_type = key.split('_')
    if len(surface_type) > 2:
        surface_type = ' '.join(surface_type[:2])
    else:
        surface_type = surface_type[0]

    # For each sensor, collect relevant columns and create a new DataFrame
    for sensor_location in sensor_locations:
        # Find all columns related to this sensor location
        sensor_columns = [col for col in trial_df.columns if sensor_location in col]

        if sensor_columns:
            # Create a new DataFrame for this sensor
            df_by_sensor = pd.DataFrame({
                "time": trial_df["time"],
                "participant_id": trial_df["participant_id"],
                "task": trial_df["task"],
                "surface_type": surface_type,
                "sensor_location": sensor_location
            })

            # For each sensor-related column, add it to the new DataFrame
            for col in sensor_columns:
                new_column_name = remove_sensor_location(col)
                df_by_sensor[new_column_name] = trial_df[col]

            # Append the sensor DataFrame to the list
            df_list.append(df_by_sensor)

    # Concatenate all DataFrames for this trial and add to the main list if any were created
    if df_list:
        trial_df_combined = pd.concat(df_list)
        all_trials_df_list.append(trial_df_combined)

        print(f"Processed {key}")
    else:
        print(f"No valid sensor columns to process for {key}")

# Concatenate all trials into a single DataFrame
all_foot_sensors_df = pd.concat(all_trials_df_list, ignore_index=True)

# Uncomment to export the final combined DataFrame to CSV
# final_combined_df.to_csv('/path/to/final_output.csv', index=False)



Processed walk_id15
Processed walk_id01
Processed walk_id12
Processed walk_id14
Processed walk_id24
Processed walk_id23
Processed walk_id08
Processed walk_id07
Processed walk_id25
Processed walk_id22
Processed walk_id19
Processed walk_id02
Processed walk_id04
Processed walk_id10
Processed walk_id16
Processed walk_id18
Processed walk_id17
Processed walk_id05
Processed walk_id13
Processed walk_id03
Processed slope_down_id15
Processed slope_down_id01
Processed slope_down_id12
Processed slope_down_id14
Processed slope_down_id24
Processed slope_down_id23
Processed slope_down_id08
Processed slope_down_id07
Processed slope_down_id25
Processed slope_down_id22
Processed slope_down_id19
Processed slope_down_id02
Processed slope_down_id04
Processed slope_down_id10
Processed slope_down_id16
Processed slope_down_id18
Processed slope_down_id17
Processed slope_down_id05
Processed slope_down_id13
Processed slope_down_id03
Processed slope_up_id15
Processed slope_up_id01
Processed slope_up_id12
Processe

In [36]:
foot_sensors_df = all_foot_sensors_df
foot_sensors_extras_df = all_foot_sensor_extras_df

In [37]:
len(foot_sensors_df['participant_id'].unique())

20

In [38]:
len(foot_sensors_extras_df['participant_id'].unique())

20

In [39]:
foot_sensors_df = foot_sensors_df.sort_values(by=['participant_id', 'time', 'sensor_location'])


In [40]:
foot_sensors_extras_df = foot_sensors_extras_df.sort_values(by=['participant_id', 'repetition', 'time'])

In [41]:
# Function to pair heel strikes with the next toe-off event
def pair_heel_strikes_toe_offs(heel_strikes, toe_offs):
    pairs = []
    toe_idx = 0
    # Loop over each heel strike
    for heel in heel_strikes:
        # Find the next toe-off that occurs after the heel strike
        while toe_idx < len(toe_offs) and toe_offs[toe_idx] < heel:
            toe_idx += 1
        if toe_idx < len(toe_offs):
            # Pair found
            pairs.append((heel, toe_offs[toe_idx]))
            toe_idx += 1  # Move to the next toe-off for the next pair
    return pairs


In [42]:
# Initialize or clear gait cycle columns for each new run
foot_sensors_extras_df['right_gait_cycle'] = np.nan
foot_sensors_extras_df['left_gait_cycle'] = np.nan

for p in foot_sensors_df['participant_id'].unique():

    current_foot_sensors_df = foot_sensors_df.loc[foot_sensors_df['participant_id'] == p]
    current_foot_sensors_extras_df = foot_sensors_extras_df.loc[foot_sensors_extras_df['participant_id'] == p]

    # Define heel strikes and toe-offs for the right foot
    right_foot_heel_strikes = current_foot_sensors_extras_df[
        (current_foot_sensors_extras_df['insoles_RightFoot_is_step'] == True) &
        (current_foot_sensors_extras_df['insoles_RightFoot_is_lifted'] == False)]
    right_foot_toe_off = current_foot_sensors_extras_df[
        (current_foot_sensors_extras_df['insoles_RightFoot_is_step'] == False) &
        (current_foot_sensors_extras_df['insoles_RightFoot_is_lifted'] == True)]

    # Define heel strikes and toe-offs for the left foot
    left_foot_heel_strikes = current_foot_sensors_extras_df[
        (current_foot_sensors_extras_df['insoles_LeftFoot_is_step'] == True) &
        (current_foot_sensors_extras_df['insoles_LeftFoot_is_lifted'] == False)]
    left_foot_toe_off = current_foot_sensors_extras_df[
        (current_foot_sensors_extras_df['insoles_LeftFoot_is_step'] == False) &
        (current_foot_sensors_extras_df['insoles_LeftFoot_is_lifted'] == True)]

    # Convert heel strikes and toe-offs to lists
    heel_strike_R = list(right_foot_heel_strikes['time'])
    toe_off_R = list(right_foot_toe_off['time'])

    heel_strike_L = list(left_foot_heel_strikes['time'])
    toe_off_L = list(left_foot_toe_off['time'])

    # Generate pairs of steps for right and left foot
    right_foot_steps = pair_heel_strikes_toe_offs(heel_strike_R, toe_off_R)
    left_foot_steps = pair_heel_strikes_toe_offs(heel_strike_L, toe_off_L)

    # Enumerate and assign step cycles to the DataFrame
    for i, (start, end) in enumerate(right_foot_steps):
        foot_sensors_extras_df.loc[
            (foot_sensors_extras_df['participant_id'] == p) &
            (foot_sensors_extras_df['time'] >= start) &
            (foot_sensors_extras_df['time'] <= end), 'right_step_count'] = i + 1

    for i, (start, end) in enumerate(left_foot_steps):
        foot_sensors_extras_df.loc[
            (foot_sensors_extras_df['participant_id'] == p) &
            (foot_sensors_extras_df['time'] >= start) &
            (foot_sensors_extras_df['time'] <= end), 'left_step_count'] = i + 1

    print(f"Processed participant {p}")


Processed participant 1
Processed participant 2
Processed participant 3
Processed participant 4
Processed participant 5
Processed participant 7
Processed participant 8
Processed participant 10
Processed participant 12
Processed participant 13
Processed participant 14
Processed participant 15
Processed participant 16
Processed participant 17
Processed participant 18
Processed participant 19
Processed participant 22
Processed participant 23
Processed participant 24
Processed participant 25


In [43]:
# show entire dataframe
# pd.set_option('display.max_rows', None)

In [44]:
# foot_sensors_extras_df.head(400)

In [45]:
# foot_sensors_extras_df.loc[(foot_sensors_extras_df['participant_id'] == 1) & (foot_sensors_extras_df['repetition'] == 3)].head(100)

In [46]:
foot_sensors_extras_df['right_gait_cycle'] = foot_sensors_extras_df['right_step_count'].fillna(method='ffill')
foot_sensors_extras_df['left_gait_cycle'] = foot_sensors_extras_df['left_step_count'].fillna(method='ffill')


<ipython-input-46-c7b87cfd3e8c>:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  foot_sensors_extras_df['right_gait_cycle'] = foot_sensors_extras_df['right_step_count'].fillna(method='ffill')
<ipython-input-46-c7b87cfd3e8c>:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  foot_sensors_extras_df['left_gait_cycle'] = foot_sensors_extras_df['left_step_count'].fillna(method='ffill')


In [47]:
# foot_sensors_extras_df.head(200)

In [48]:
# merge the foot_sensors_extras_df left onto foot_sensors_df on participant_id, time
foot_sensors_df_gait_cycles = pd.merge(foot_sensors_df, foot_sensors_extras_df[['participant_id', 'time', 'right_step_count', 'left_step_count', 'right_gait_cycle', 'left_gait_cycle']], on=['participant_id', 'time'], how='left')

In [49]:
# foot_sensors_df_gait_cycles.head(100)

In [50]:
# export csv to folder
# all_foot_sensors_df.to_csv('/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB_extras/foot_pressure_data/all_foot_sensors_df.csv', index=False)

In [51]:
# export csv to folder
# all_foot_sensor_extras_df.to_csv('/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB_extras/foot_pressure_data/all_foot_sensor_extras_df.csv', index=False)

In [52]:
foot_sensors_df_gait_cycles.to_csv('/content/drive/MyDrive/Next_Step/Honda_Participants/CourseB_extras/foot_pressure_data/all_foot_sensor_df.csv', index=False)